# Init
---

In [1]:
#### mv packages ####
import modules.data as d
import modules.model as m
import modules.pooling as p
import modules.train as t
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device(['cuda:1', 'cuda:0'])

#### data ####
brca = d.Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=dataset_dir/'tcga',
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=True, 
    log_transform=True,
    scale_method='standard',

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)
_dataset = d.GraphDataset(brca)
_batch = d.get_toy_databatch(_dataset, generator)

#### convenience variables ####

# predefined
_embedding_size = 128

# from mask (init)
_mask = brca.mask
_num_nodes, _num_sets = _mask.shape

# from batch (forward)
_batch_size = int(_batch.x.shape[0]/_num_nodes)
_num_node_features = _batch.x.shape[1] # or brca.num_node_features

# #### Device() ####
# device = cuda:4

# #### Preprocessor() ####
# log0_method              log1p                    str
# class_weights            (6,)                     Tensor (cuda:4)
# edge_index               (2, 32798)               Tensor (cuda:4)
# edge_attr                (32798, 16)              Tensor (cuda:4)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# mask_list                305                      list
# mask                     (4383, 305)              Tensor (cuda:4)
# x                        (562, 4383, 1)           Tensor (cuda:4)
# y                        (562,)                   Tensor (cuda:4)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int


# Transformer Attention
---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from typing import Optional

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:Optional[float]=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.embed_dim, self.head_dim = self._get_dims(embed_dim, head_dim)
        self.scale_factor = self.head_dim ** 0.5 # sqrt(d_k)
        
        self.lin_qkv = nn.Linear(self.embed_dim, 3 * self.num_heads * self.head_dim)
        self.lin_out = nn.Linear(self.num_heads * self.head_dim, self.embed_dim)
        self.dropout = nn.Dropout(p=dropout) if dropout else None
        
    def _get_dims(self, embed_dim:int=None, head_dim:int=None):
        # none specified; assert error
        assert (embed_dim is not None) or (head_dim is not None), 'one of [embed_dim, head_dim] must be specified'

        # both specified; lin_out reshapes head to embed
        if (embed_dim is not None) and (head_dim is not None):
            return embed_dim, head_dim

        # embed_dim specified; head = embed / num_heads
        elif embed_dim is not None:
            assert embed_dim % self.num_heads == 0, 'embed_dim must be divisible by self.num_heads'
            head_dim = embed_dim // self.num_heads

        # head_dim specified; embed = head * num_heads
        elif head_dim is not None:
            embed_dim = head_dim * self.num_heads

        return embed_dim, head_dim

    def _expand_mask(self, mask:Tensor, batch_size:int):
        if mask.dim() == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
            mask = mask.expand(batch_size, self.num_heads, -1, -1)
        elif mask.dim() == 3:
            mask = mask.unsqueeze(1)
            mask = mask.expand(-1, self.num_heads, -1, -1)
        else:
            raise ValueError(f'mask shape unsupported: {mask.shape}')
        return mask.bool()

    def forward(self, x:Tensor, mask:Optional[Tensor]=None, return_weight:bool=False):
        batch_size, num_nodes, _ = x.shape

        # get qkv
        qkv = self.lin_qkv(x).reshape(batch_size, num_nodes, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(0,3,1,2,4) # (batch_size(0), num_heads(3), num_nodes(1), qkv(2), head_dim(4))
        q, k, v = qkv[...,0,:], qkv[...,1,:], qkv[...,2,:] # split into (batch_size, num_heads, num_nodes, head_dim) for bmm

        # compute attention weight
        weight = (q @ k.transpose(-2,-1))/self.scale_factor # (batch_size, num_heads, num_nodes, num_nodes)

        # mask if provided
        if mask is not None:
            mask = self._expand_mask(mask, batch_size)
            weight = weight.masked_fill(~mask, float('-inf'))

        # softmax
        weight = F.softmax(weight, dim=-1)

        # apply dropout
        if self.dropout is not None:
            weight = self.dropout(weight)

        # compute, format output
        out = weight @ v # (batch_size, num_heads, num_nodes, head_dim)
        out = out.permute(0,2,1,3) # (batch_size(0), num_nodes(2), num_heads(1), head_dim(3))
        out = out.reshape(batch_size, num_nodes, self.num_heads * self.head_dim) # (batch_size, num_nodes, num_heads * head_dim)

        # project output from heads (batch_size, num_nodes, embed_dim)
        out = self.lin_out(out) 

        return out, weight if return_weight else out


In [ ]:
class CrossAttention(SelfAttention):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:Optional[float]=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.embed_dim, self.head_dim = self._get_dims(embed_dim, head_dim)
        self.scale_factor = self.head_dim ** 0.5 # sqrt(d_k)
        
        self.lin_q = nn.Linear(self.embed_dim, self.num_heads * self.head_dim)
        self.lin_kv = nn.Linear(self.embed_dim, 2 * self.num_heads * self.head_dim)
        self.lin_out = nn.Linear(self.num_heads * self.head_dim, self.embed_dim)
        self.dropout = nn.Dropout(p=dropout) if dropout else None
        
    def forward(self, query:Tensor, context:Tensor, mask:Optional[Tensor]=None, return_weight:bool=False):
        batch_size, num_query, _ = query.shape
        _, num_context, _ = context.shape

        # get qkv
        q = self.lin_q(query).reshape(batch_size, num_query, self.num_heads, self.head_dim)
        kv = self.lin_kv(context).reshape(batch_size, num_context, 2, self.num_heads, self.head_dim)
        kv = kv.permute(0,3,1,2,4) # (batch_size(0), num_heads(3), num_nodes(1), qkv(2), head_dim(4))
        k, v = kv[...,0,:], kv[...,1,:] # split into (batch_size, num_heads, num_nodes, head_dim) for bmm

        # compute attention weight
        weight = (q @ k.transpose(-2,-1))/self.scale_factor # (batch_size, num_heads, num_nodes, num_nodes)

        # mask if provided
        if mask is not None:
            mask = self._expand_mask(mask, batch_size)
            weight = weight.masked_fill(~mask, float('-inf'))

        # softmax
        weight = F.softmax(weight, dim=-1)

        # apply dropout
        if self.dropout is not None:
            weight = self.dropout(weight)

        # compute, format output
        out = weight @ v # (batch_size, num_heads, num_nodes, head_dim)
        out = out.permute(0,2,1,3) # (batch_size(0), num_nodes(2), num_heads(1), head_dim(3))
        out = out.reshape(batch_size, num_query, self.num_heads * self.head_dim) # (batch_size, num_nodes, num_heads * head_dim)

        # project output from heads (batch_size, num_nodes, embed_dim)
        out = self.lin_out(out) 

        return out, weight if return_weight else out


In [ ]:
class MultiQueryAttention(SelfAttention):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:Optional[float]=0.1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.embed_dim, self.head_dim = self._get_dims(embed_dim, head_dim)
        self.scale_factor = self.head_dim ** 0.5 # sqrt(d_k)
        
        self.lin_q = nn.Linear(self.embed_dim, self.num_heads * self.head_dim)
        self.lin_kv = nn.Linear(self.embed_dim, 2 * self.head_dim) # shared across heads
        self.lin_out = nn.Linear(self.num_heads * self.head_dim, self.embed_dim)
        self.dropout = nn.Dropout(p=dropout) if dropout is not None else None
        
    def forward(self, query:Tensor, context:Tensor, mask:Optional[Tensor]=None, return_weight:bool=False):
        batch_size, num_query, _ = query.shape
        _, num_context, _ = context.shape

        # get qkv
        q = self.lin_q(query).reshape(batch_size, num_query, self.num_heads, self.head_dim)
        k, v = self.lin_kv(context).chunk(2, dim=-1)
        k = k.unsqueeze(1).expand(batch_size, self.num_heads, num_context, self.head_dim)
        v = v.unsqueeze(1).expand(batch_size, self.num_heads, num_context, self.head_dim)

        # compute attention weight
        weight = (q @ k.transpose(-2,-1))/self.scale_factor # (batch_size, num_heads, num_nodes, num_nodes)

        # mask if provided
        if mask is not None:
            mask = self._expand_mask(mask, batch_size)
            weight = weight.masked_fill(~mask, float('-inf'))

        # softmax
        weight = F.softmax(weight, dim=-1)

        # apply dropout
        if self.dropout is not None:
            weight = self.dropout(weight)

        # compute, format output
        out = weight @ v # (batch_size, num_heads, num_nodes, head_dim)
        out = out.permute(0,2,1,3) # (batch_size(0), num_nodes(2), num_heads(1), head_dim(3))
        out = out.reshape(batch_size, num_query, self.num_heads * self.head_dim) # (batch_size, num_nodes, num_heads * head_dim)

        # project output from heads (batch_size, num_nodes, embed_dim)
        out = self.lin_out(out) 

        return out, weight if return_weight else out
